# Chapter 9
## Spike Frequency Adaptation
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter09.ipynb)

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact
from mnd.core import alpha_h, alpha_m, alpha_n, beta_h, beta_m, beta_n, h_inf, m_inf, n_inf


## A Model M-Current

Steady-state activation $w_\infty(v)$ and time constant $\tau_w(v)$ for the slow M-current gate used by the RTM+M-current neuron below.

In [ ]:
def w_inf(v):
    return 1.0 / (1.0 + np.exp(-(v + 35.0) / 10.0))


def tau_w(v):
    return 400.0 / (3.3 * np.exp((v + 35.0) / 20.0) + np.exp(-(v + 35.0) / 20.0))


def m_current(v=None):
    if v is None:
        v = np.arange(-100, 51)
    return v, w_inf(v), tau_w(v)


In [ ]:
v, wi, tw = m_current()
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(v, wi, color='k', linewidth=2)
ax[0].set_xlabel('$v$ [mV]')
ax[0].set_ylabel(r'$w_\infty$')
ax[1].plot(v, tw, color='k', linewidth=2)
ax[1].set_xlabel('$v$ [mV]')
ax[1].set_ylabel(r'$\tau_w$')
plt.tight_layout()
plt.show()


## RTM Neuron with an M-Current

`m` is not its own state -- the book's `RTM_M/make_figure.m` always sets `m = m_inf(v)` (quasi-static), never integrates it as an ODE.

In [ ]:
def simulate_rtm_m(i_ext=1.5, g_m=0.25, t_final=300, dt=0.01):
    g_k, g_na, g_l = 80, 100, 0.1
    v_k, v_na, v_l = -100, 50, -67

    def derivative(x0, t):
        v, n, h, w = x0
        m = m_inf(v)
        dv = (i_ext - g_na * h * m ** 3 * (v - v_na) - g_k * n ** 4 * (v - v_k)
              - g_l * (v - v_l) - g_m * w * (v - v_k))
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
        dw = (w_inf(v) - w) / tau_w(v)
        return [dv, dn, dh, dw]

    v0 = -70.0
    x0 = [v0, n_inf(v0), h_inf(v0), 0.0]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    return t, sol[:, 0], sol[:, 3]


def plot_rtm_m(t, v, w):
    fig, ax = plt.subplots(2, figsize=(7, 6), sharex=True)
    ax[0].plot(t, v, lw=2, c='k')
    ax[0].set_ylabel('v [mV]')
    ax[1].plot(t, w, lw=2, c='k')
    ax[1].set_xlabel('t [ms]')
    ax[1].set_ylabel('w')
    ax[1].set_ylim(0, max(w) * 1.2)
    ax[0].set_xlim(min(t), max(t))
    plt.tight_layout()
    plt.show()


In [ ]:
plot_rtm_m(*simulate_rtm_m())


In [ ]:
interact(lambda i_ext=1.5, g_m=0.25: plot_rtm_m(*simulate_rtm_m(i_ext=i_ext, g_m=g_m)),
         i_ext=(0.0, 3.0, 0.1), g_m=(0.0, 1.0, 0.05));


### RTM Neuron with an M-Current, at Rest ($I_{ext}=0$)

In [ ]:
plot_rtm_m(*simulate_rtm_m(i_ext=0, t_final=600))


## Calcium-Dependent AHP Currents

Voltage-dependent calcium target $c_\infty(v)$ used by the calcium-dependent AHP current below.

In [ ]:
def cac_inf(v):
    return (120 - v) / (1 + np.exp(-(v + 15) / 5)) * 4 / 25


def calcium_rise(v=None):
    if v is None:
        v = np.arange(-1000, 501) / 10.
    return v, cac_inf(v)


In [ ]:
v, r = calcium_rise()
plt.figure(figsize=(7, 4))
plt.plot(v, r, color='k', linewidth=2)
plt.xlabel('$v$ [mV]')
plt.ylabel(r'$c_\infty$')
plt.tight_layout()
plt.show()


## RTM Neuron with a Calcium-Dependent AHP Current

As with the M-current model, `m` is quasi-static (`m = m_inf(v)`), and the calcium time constant is a plain constant (80), not voltage-dependent.

In [ ]:
def simulate_rtm_ahp(i_ext=1.5, g_ahp=0.25, t_final=300, dt=0.01):
    g_k, g_na, g_l = 80, 100, 0.1
    v_k, v_na, v_l = -100, 50, -67

    def derivative(x0, t):
        v, n, h, ca = x0
        m = m_inf(v)
        dv = (i_ext - g_na * h * m ** 3 * (v - v_na) - g_k * n ** 4 * (v - v_k)
              - g_l * (v - v_l) - g_ahp * ca * (v - v_k))
        dn = alpha_n(v) * (1.0 - n) - beta_n(v) * n
        dh = alpha_h(v) * (1.0 - h) - beta_h(v) * h
        dca = (cac_inf(v) - ca) / 80.0
        return [dv, dn, dh, dca]

    v0 = -70.0
    x0 = [v0, n_inf(v0), h_inf(v0), 0.0]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative, x0, t)
    return t, sol[:, 0], sol[:, 3]


def plot_rtm_ahp(t, v, ca):
    fig, ax = plt.subplots(2, figsize=(7, 6), sharex=True)
    ax[0].plot(t, v, lw=2, c='k')
    ax[0].set_ylabel('v [mV]')
    ax[1].plot(t, ca, lw=2, c='k')
    ax[1].set_xlabel('t [ms]')
    ax[1].set_ylabel('$[Ca^{2+}]$')
    ax[1].set_ylim(0, max(ca) * 1.2)
    ax[0].set_xlim(min(t), max(t))
    plt.tight_layout()
    plt.show()


In [ ]:
plot_rtm_ahp(*simulate_rtm_ahp())


In [ ]:
interact(lambda i_ext=1.5, g_ahp=0.25: plot_rtm_ahp(*simulate_rtm_ahp(i_ext=i_ext, g_ahp=g_ahp)),
         i_ext=(0.0, 3.0, 0.1), g_ahp=(0.0, 1.0, 0.05));


### RTM Neuron with AHP, at Rest ($I_{ext}=0$)

In [ ]:
plot_rtm_ahp(*simulate_rtm_ahp(i_ext=0, t_final=600))


## LIF Neuron with Spike-Triggered Adaptation

Dimensionless LIF model (same convention as chapter 7), with an adaptation current `w` that jumps by `delta` at every spike and decays with time constant `tau_a`.

In [ ]:
def simulate_lif_adapt(tau_m=10., I=0.13, tau_a=40., delta=0.05, t_final=300., dt=0.01):
    def derivative(state):
        v, w = state
        dv = -v / tau_m + I - w * v
        dw = -w / tau_a
        return np.array([dv, dw])

    def integrate_rk4(x, dt, f):
        k1 = dt * f(x)
        k2 = dt * f(x + 0.5 * k1)
        k3 = dt * f(x + 0.5 * k2)
        k4 = dt * f(x + k3)
        return x + (k1 + 2. * (k2 + k3) + k4) / 6.

    num_steps = int(t_final / dt)
    t = np.arange(0, t_final, dt)
    v = np.zeros(num_steps)
    w = np.zeros(num_steps)

    for i in range(1, num_steps):
        v_new, w_new = integrate_rk4(np.array([v[i - 1], w[i - 1]]), dt, derivative)
        if v_new <= 1:
            v[i] = v_new
            w[i] = w_new
        else:
            v[i] = 0.
            w[i] = w_new + delta

    return t, v, w


def plot_lif_adapt(t, v, w):
    fig, ax = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    ax[0].plot(t, v, color='k', linewidth=2)
    ax[0].set_ylabel('$v$')
    ax[0].set_ylim(0, 4)
    ax[0].set_yticks([0, 1])
    ax[1].plot(t, w, color='k', linewidth=2)
    ax[1].set_xlabel('$t$')
    ax[1].set_ylabel('$w$')
    ax[1].set_xlim(0, max(t))
    plt.tight_layout()
    plt.show()


In [ ]:
plot_lif_adapt(*simulate_lif_adapt())


In [ ]:
interact(lambda I=0.13, tau_a=40., delta=0.05: plot_lif_adapt(*simulate_lif_adapt(I=I, tau_a=tau_a, delta=delta)),
         I=(0.05, 0.3, 0.01), tau_a=(10., 100., 5.), delta=(0.0, 0.2, 0.01));


## Spike-to-Spike Adaptation Map

$\phi(z)$: the adaptation value just after the next spike, as a function of $z$, its value just after the previous spike. This is an event/threshold-crossing map (Heun/RK2 scheme), not a smooth ODE.

In [ ]:
def adaptation_map(tau_m=10., I=0.12, tau_w=100., delta=0.01, z_max=0.05, dt=0.01, N=100):
    dt05 = dt / 2
    v = np.zeros(N + 1)
    w = np.arange(N + 1) / N * z_max
    phi = np.full(N + 1, np.nan)
    done = np.zeros(N + 1, dtype=bool)

    # I <= 1/tau_m never reaches threshold (v asymptotes to tau_m*I <= 1),
    # so the loop needs a hard cap -- undone entries stay nan (plotted as a gap).
    max_steps = 200_000
    for _ in range(max_steps):
        if done.all():
            break
        v_old = v.copy()
        w_old = w.copy()
        v_inc = -v / tau_m + I - w * v
        w_inc = -w / tau_w
        v_tmp = v + dt05 * v_inc
        w_tmp = w + dt05 * w_inc
        v_inc = -v_tmp / tau_m + I - w_tmp * v_tmp
        w_inc = -w_tmp / tau_w
        v = v + dt * v_inc
        w = w + dt * w_inc

        ind = (v > 1) & (~done)
        done[ind] = True
        phi[ind] = (v[ind] - 1) * w_old[ind] + (1 - v_old[ind]) * w[ind]
        phi[ind] = phi[ind] / (v[ind] - v_old[ind]) + delta

    z = np.arange(N + 1) / N * z_max
    return z, phi


def plot_adaptation_map(z, phi, z_max=None):
    if z_max is None:
        z_max = z[-1]
    plt.figure(figsize=(6, 6))
    plt.plot(z, phi, color='k', linewidth=2)
    plt.plot([0, z_max], [0, z_max], color='k', linestyle='dashed')
    plt.xlim(0, z_max)
    plt.ylim(0, z_max)
    plt.gca().set_aspect('equal')
    plt.xlabel('$z$')
    plt.ylabel(r'$\phi(z)$')
    plt.tight_layout()
    plt.show()


In [ ]:
plot_adaptation_map(*adaptation_map())


In [ ]:
interact(lambda I=0.12, delta=0.01: plot_adaptation_map(*adaptation_map(I=I, delta=delta)),
         I=(0.11, 0.3, 0.01), delta=(0.0, 0.05, 0.005));


## $v$ and its Adaptation-Free Counterpart $\tilde v$

A purely subthreshold comparison: $v$ has an adaptation-like multiplicative term $w_k e^{-t/\tau_w} v$ that decays over time, shown against a version with a larger initial $\tilde w_k$. No threshold or reset here -- both stay subthreshold throughout.

In [ ]:
def v_v_tilde(tau_m=10., I=0.13, w_k=0.05, tilde_w_k=0.08, tau_w=40., t_final=100., dt=0.01):
    def derivative(v, t, wk):
        return -v / tau_m + I - wk * np.exp(-t / tau_w) * v

    t = np.arange(0, t_final + dt, dt)
    v = odeint(derivative, 0., t, args=(w_k,))[:, 0]
    v_tilde = odeint(derivative, 0., t, args=(tilde_w_k,))[:, 0]
    return t, v, v_tilde


def plot_v_v_tilde(t, v, v_tilde):
    plt.figure(figsize=(7, 4))
    plt.plot(t, v, color='k', linewidth=2, label='$v$')
    plt.plot(t, v_tilde, color='k', linewidth=2, linestyle='dashed', label=r'$\tilde{v}$')
    plt.xlabel('$t$')
    plt.title(r'$v$ (solid) and $\tilde{v}$ (dashes)')
    plt.xlim(0, 60)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_v_v_tilde(*v_v_tilde())


In [ ]:
interact(lambda w_k=0.05, tilde_w_k=0.08: plot_v_v_tilde(*v_v_tilde(w_k=w_k, tilde_w_k=tilde_w_k)),
         w_k=(0.0, 0.2, 0.01), tilde_w_k=(0.0, 0.2, 0.01));
